# Retraining after a policy censors labels — simulation

A deployed policy prevents some outcomes from being observed. This notebook asks which
retraining strategy is safest once the resulting labels are no longer missing at random.

We simulate six interpretable operating regimes at one and twelve months after launch. Every
strategy sees the same six-month rolling window, model class, features and paired population
draw. The metric is normalized pAUC at 20% FPR: random is 0.1 and perfect ranking is 1.0.

In [0]:
%pip install lightgbm --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import importlib
import importlib.metadata
import json
import os
import platform
import sys

import pandas as pd

try:
    _path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    HERE = "/Workspace" + "/".join(_path.split("/")[:-1])
except NameError:
    HERE = os.getcwd()

if HERE not in sys.path:
    sys.path.insert(0, HERE)

import censoring_sim as sim
import sim_core as core
import study

importlib.reload(core)
importlib.reload(sim)
importlib.reload(study)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("Study modules loaded from the notebook directory.")
print(json.dumps({
    "python": platform.python_version(),
    **{package: importlib.metadata.version(package) for package in (
        "numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "joblib",
        "matplotlib",
    )},
}, indent=2))

Study modules loaded from the notebook directory.
{
  "python": "3.11.11",
  "numpy": "1.23.5",
  "pandas": "1.5.3",
  "scipy": "1.11.1",
  "scikit-learn": "1.3.0",
  "lightgbm": "4.3.0",
  "joblib": "1.2.0",
  "matplotlib": "3.7.2"
}


## Design

The reference is a 1% event rate, 6% trigger rate and 5% randomized holdout. The panel then
varies the regional signal hidden by the policy, regional drift, trigger rate, holdout size
and base rate. There are no launch ramps, delayed shifts or policy-information lags.

In [0]:
SCENARIOS = study.scenario_frame().copy()
display_columns = [
    "scenario_id", "label", "region_shift", "region_drift_per_month",
    "trigger_rate", "holdout_pct", "base_rate", "drift_per_month",
]
print(SCENARIOS[display_columns].to_string(index=False))

print("\nTimeline and scale:")
print(json.dumps({
    "periods_months_post_launch": study.PERIODS,
    "seeds": list(study.SEEDS),
    "rows_per_month": study.BASE["rows_per_month"],
    "training_window_months": study.BASE["window_months"],
    "validation_rows": study.BASE["n_valid"],
    "test_rows": study.BASE["n_test"],
}, indent=2))

             scenario_id                             label  region_shift  region_drift_per_month  trigger_rate  holdout_pct  base_rate  drift_per_month
                no_shift                 No regional shift          0.00                    0.00          0.06         0.05      0.010             0.02
               low_shift        Low shift + regional drift          0.25                    0.02          0.06         0.05      0.010             0.02
               reference                         Reference          1.50                    0.00          0.06         0.05      0.010             0.02
         reference_drift        Reference + regional drift          1.50                    0.02          0.06         0.05      0.010             0.02
high_trigger_low_holdout No shift; 12% trigger; 2% holdout          0.00                    0.00          0.12         0.02      0.010             0.02
   low_base_high_holdout       0.1% base rate; 20% holdout          1.50                

## Diagnostic experiments

Two diagnostics establish why the method comparison is necessary:

1. **Drift trajectory:** fit once on all pre-launch data, then score the unchanged model every
   month from launch through month 12 under four global-drift rates. Regional shift is fixed
   to zero so this isolates temporal drift.
2. **Metric bias:** every fitted method is scored on both the randomized holdout and the
   observable unflagged validation population. Their difference shows when an apparently good
   production metric is biased by the policy.

In [0]:
print("Global drift rates:", study.DRIFT_LEVELS)
print("Metric-bias columns: observed_valid_pAUC, holdout_valid_pAUC")

Global drift rates: (0.0, 0.01, 0.02, 0.04)
Metric-bias columns: observed_valid_pAUC, holdout_valid_pAUC


## Methods

R0 is an unavailable uncensored benchmark. R1–R4 and R6–R7 are actual fitted models. R5 is
Asymmetric IPW: pooled validation performance chooses either dropping (`α = holdout_pct`) or
pure IPW (`α = 1`); R5 then copies that selected endpoint. It adds no fit
and cannot switch methods inside a replicate.

In [0]:
print(pd.DataFrame({
    "method": study.METHOD_ORDER,
    "method_label": [study.METHOD_LABELS[method] for method in study.METHOD_ORDER],
}).to_string(index=False))

            method         method_label
      R0_benchmark uncensored benchmark
   R1_holdout_only         holdout only
 R2_unflagged_only           no holdout
R3_drop_unweighted             dropping
       R4_ipw_pure                  IPW
     R5_asymmetric       Asymmetric IPW
     R6_no_retrain        no retraining
    R7_incremental          incremental


## Run or resume

Each scenario-period-seed task is checkpointed before aggregation. Rerunning skips
completed tasks.

In [0]:
N_WORKERS = int(os.environ.get("SIM_WORKERS", 5))
THREADS_PER_WORKER = int(os.environ.get("SIM_THREADS_PER_WORKER", 12))

RESULT = study.run_study(
    HERE,
    out_name="sim-results",
    n_workers=N_WORKERS,
    threads_per_worker=THREADS_PER_WORKER,
    seeds=study.SEEDS,
    dgp_seed=0,
)
DRIFT = study.run_drift_study(
    HERE,
    out_name="sim-results",
    n_workers=N_WORKERS,
    threads_per_worker=THREADS_PER_WORKER,
    seeds=study.SEEDS,
    dgp_seed=0,
)

progress = study._read_json(os.path.join(RESULT["out_dir"], "progress.json"))
drift_progress = study._read_json(os.path.join(RESULT["out_dir"], "drift_progress.json"))
print(json.dumps(progress, indent=2))
print(json.dumps(drift_progress, indent=2))
if (progress["stage"] != "complete" or progress["errors"]
        or drift_progress["stage"] != "complete" or drift_progress["errors"]):
    raise RuntimeError("simulation diagnostics did not finish cleanly")

{
  "completed": 60,
  "elapsed_minutes": 47.16970172723134,
  "errors": 0,
  "stage": "complete",
  "total": 60
}
{
  "completed": 20,
  "elapsed_minutes": 8.161759773890177,
  "errors": 0,
  "stage": "complete",
  "total": 20
}


## Integrity readout

These are checks on the produced evidence, not the conclusions. `IPW − dropping` is paired
within seed and classified using a two-sided 95% t interval. Asymmetric-IPW regret uses one
fixed better endpoint per scenario-period.

In [0]:
endpoints = RESULT["endpoint_summary"]
asymmetric = RESULT["asymmetric_summary"]
methods = RESULT["method_summary"]

print("Endpoint comparison:")
print(endpoints[[
    "label", "period", "n", "ipw_minus_dropping", "paired_sd",
    "lower_95", "upper_95", "negative_seeds", "positive_seeds", "result",
]].round(6).to_string(index=False))

print("\nAsymmetric IPW selection:")
print(asymmetric[[
    "label", "period", "selected_endpoint", "better_endpoint",
    "gap_vs_better_endpoint", "lower_95", "upper_95", "result_vs_endpoint",
    "best_other_method", "gap_vs_best_other", "asymmetric_rank",
]].round(6).to_string(index=False))

print("\nRows written:")
for name in (
    "scenario_definitions", "method_raw", "method_summary", "endpoint_summary",
    "asymmetric_summary", "drift_trajectory",
):
    path = os.path.join(RESULT["out_dir"], f"{name}.csv")
    frame = pd.read_csv(path)
    print(f"{name:24s} {len(frame):4d} rows  sim-results/{name}.csv")

Endpoint comparison:
                            label period  n  ipw_minus_dropping  paired_sd  lower_95  upper_95  negative_seeds  positive_seeds        result
No shift; 12% trigger; 2% holdout  early  5            0.000600   0.001524 -0.001292  0.002492               1               4           tie
No shift; 12% trigger; 2% holdout   late  5           -0.014921   0.002485 -0.018007 -0.011835               5               0 dropping_wins
      0.1% base rate; 20% holdout  early  5           -0.000625   0.004682 -0.006438  0.005188               3               2           tie
      0.1% base rate; 20% holdout   late  5            0.004479   0.004697 -0.001353  0.010310               0               5           tie
       Low shift + regional drift  early  5           -0.000196   0.002270 -0.003015  0.002623               3               2           tie
       Low shift + regional drift   late  5           -0.006609   0.000890 -0.007713 -0.005504               5               0 droppi

`02_results` reads only the six aggregate CSVs above. Checkpoints are resumability artifacts
and can be removed before publication after both notebooks have been reproduced once.